# Exploratory Analysis of app data

- The goal of this project is to understand what common attributes profitable apps share among the Apple store and Google Play Store

In [161]:
# Imports
import numpy as np
import pandas as pd
from csv import reader

In [162]:
# Creating a function to help explore the datasets
def explore_data(dataset, start, end, rows_and_columns=False):
    dataset_slice = dataset[start:end]
    for row in dataset_slice:
        print(row)
        print("\n")

    if rows_and_columns:
        print("Number of rows: ", len(dataset_slice))
        print("Number of columns: ", len(dataset_slice[0]))

## Opening Datasets

In [163]:
# Opening both files
opened_gplay_file = open('data/googleplaystore.csv')
opened_apple_file = open('data/applestore.csv')

# Reading files
read_gplay = reader(opened_gplay_file)
read_apple = reader(opened_apple_file)

### Apple Store dataset ###
ios = list(read_apple)
ios_header = ios[0]
ios = ios[1:]

### Google Play dataset ###
gplay = list(read_gplay)
gplay_header = gplay[0]
gplay = gplay[1:]

## Using explore_data() to examine the files

In [164]:
print(ios_header)
explore_data(ios, 0, 3, True)

['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']
['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1']


['389801252', 'Instagram', '113954816', 'USD', '0.0', '2161558', '1289', '4.5', '4.0', '10.23', '12+', 'Photo & Video', '37', '0', '29', '1']


['529479190', 'Clash of Clans', '116476928', 'USD', '0.0', '2130805', '579', '4.5', '4.5', '9.24.12', '9+', 'Games', '38', '5', '18', '1']


Number of rows:  3
Number of columns:  16


In [165]:
print(gplay_header)
explore_data(gplay, 0, 3, True)

['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']
['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


['Coloring book moana', 'ART_AND_DESIGN', '3.9', '967', '14M', '500,000+', 'Free', '0', 'Everyone', 'Art & Design;Pretend Play', 'January 15, 2018', '2.0.0', '4.0.3 and up']


['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'ART_AND_DESIGN', '4.7', '87510', '8.7M', '5,000,000+', 'Free', '0', 'Everyone', 'Art & Design', 'August 1, 2018', '1.2.4', '4.0.3 and up']


Number of rows:  3
Number of columns:  13


## Some findings

The Google Play store's header looks like this:

['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']

The Apple Store's header looks like this:

['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']

- Each store records different aspects about their apps, therefore a different approach to analysis will be required for either store
- Some common columns among both stores will help with the analysis: Genre, rating/user_rating, price,

## Cleaning data
- Will now use pandas to clean both datasets
- Will get of null values, values not in English, etc.

In [166]:
gplay_df = pd.read_csv("data/googleplaystore.csv")
ios_df = pd.read_csv("data/AppleStore.csv")

## Removing Incorrect Row Entry
- Row 10472 in the Google Play dataset has an error
- That row must be removed before proceeding 

Error is identified in following discussion link: https://www.kaggle.com/datasets/lava18/google-play-store-apps/discussion?sort=undefined


In [167]:
print(gplay_df.iloc[10472])

App               Life Made WI-Fi Touchscreen Photo Frame
Category                                              1.9
Rating                                               19.0
Reviews                                              3.0M
Size                                               1,000+
Installs                                             Free
Type                                                    0
Price                                            Everyone
Content Rating                                        NaN
Genres                                  February 11, 2018
Last Updated                                       1.0.19
Current Ver                                    4.0 and up
Android Ver                                           NaN
Name: 10472, dtype: object


In [168]:
# Removing bad row
# The issue is that this row was describing rating as 19, but rating can only go up to a max of 5
gplay_df.iloc[10472].name
gplay_df = gplay_df.drop(gplay_df.index[10472])

In [169]:
# Verifying that the bad row is gone
print(gplay_df.iloc[10472])

App               osmino Wi-Fi: free WiFi
Category                            TOOLS
Rating                                4.2
Reviews                            134203
Size                                 4.1M
Installs                      10,000,000+
Type                                 Free
Price                                   0
Content Rating                   Everyone
Genres                              Tools
Last Updated               August 7, 2018
Current Ver                       6.06.14
Android Ver                    4.4 and up
Name: 10473, dtype: object


## Removing duplicate values

In [170]:
print("Google Play Duplicates: ", gplay_df.duplicated().sum())
print("Apple Store Duplicates: ", ios_df.duplicated().sum())

Google Play Duplicates:  483
Apple Store Duplicates:  0


In [171]:
gplay_df.drop_duplicates(inplace=True)

In [172]:
ios_df.drop_duplicates(inplace=True)

In [173]:
# Verifying that there are no more duplicates

print("Google Play Duplicates: ", gplay_df.duplicated().sum())
print("Apple Store Duplicates: ", ios_df.duplicated().sum())

Google Play Duplicates:  0
Apple Store Duplicates:  0


## Remove non-English rows
- Removing apps where names or information is written in a language other than English

In [174]:
# Making a function to identify English
def is_English(string):

    non_ascii = 0

    for c in string:
        if ord(c) > 127:
            non_ascii += 1 

    if non_ascii > 3: # This method will allow apps with emojis in their names to remain in the dataset
        return False
    else:
        return True




Applying function to both datasets

In [175]:
# In the Apple store dataset, the app names are under the column "track_name"
# In the Android dataset, the app names are under the column "App"

mask = ios_df['track_name'].apply(is_English)
ios_df = ios_df[mask].copy()
ios_df = ios_df.reset_index(drop=True)
ios_df.shape

(6183, 16)

In [176]:
mask = gplay_df['App'].apply(is_English)
gplay_df = gplay_df[mask].copy()
gplay_df = gplay_df.reset_index(drop=True)
gplay_df.shape

(10312, 13)

## Dropping NaN values
- Any entries that do not give information will not be helpful during the analysis

In [177]:
gplay_df.dropna()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10305,FR Calculator,FAMILY,4.0,7,2.6M,500+,Free,0,Everyone,Education,"June 18, 2017",1.0.0,4.1 and up
10307,Sya9a Maroc - FR,FAMILY,4.5,38,53M,"5,000+",Free,0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10308,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6M,100+,Free,0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10310,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,"1,000+",Free,0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [178]:
ios_df.dropna()

,id,track_name,size_bytes,currency,price,rating_count_tot,rating_count_ver,user_rating,user_rating_ver,ver,cont_rating,prime_genre,sup_devices.num,ipadSc_urls.num,lang.num,vpp_lic
0,284882215,Facebook,389879808,USD,0.00,2974676,212,3.5,3.5,95.0,4+,Social Networking,37,1,29,1
1,389801252,Instagram,113954816,USD,0.00,2161558,1289,4.5,4.0,10.23,12+,Photo & Video,37,0,29,1
2,529479190,Clash of Clans,116476928,USD,0.00,2130805,579,4.5,4.5,9.24.12,9+,Games,38,5,18,1
3,420009108,Temple Run,65921024,USD,0.00,1724546,3842,4.5,4.0,1.6.2,9+,Games,40,5,1,1
4,284035177,Pandora - Music & Radio,130242560,USD,0.00,1126879,3594,4.0,4.5,8.4.1,12+,Music,37,4,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6178,1070854722,Be-be-bears!,480781312,USD,2.99,0,0,0.0,0.0,1.0.2.5,4+,Games,35,5,13,1
6179,1169971902,Hey Duggee: We Love Animals,136347648,USD,2.99,0,0,0.0,0.0,1.2,4+,Games,40,5,1,1
6180,1170406182,Shark Boom - Challenge Friends with your Pet,245415936,USD,0.00,0,0,0.0,0.0,1.0.9,4+,Games,38,5,1,1
6181,1070052833,Go!Go!Cat!,91468800,USD,0.00,0,0,0.0,0.0,1.1.2,12+,Games,37,2,2,1


## Isolating Free Apps
- In both datasets, there is a column named 'price' 

In [179]:
print(ios_df.shape)
ios_free_df = ios_df[ios_df['price'].eq(0)].copy()
ios_free_df.shape

(6183, 16)


(3222, 16)

In [186]:
print(gplay_df.shape)
gplay_free_df = gplay_df[gplay_df['Type'].eq('Free')].copy()
gplay_free_df.head(60)

(9550, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
5,Paper flowers instructions,ART_AND_DESIGN,4.4,167,5.6M,"50,000+",Free,0,Everyone,Art & Design,"March 26, 2017",1.0,2.3 and up
6,Smoke Effect Photo Maker - Smoke Editor,ART_AND_DESIGN,3.8,178,19M,"50,000+",Free,0,Everyone,Art & Design,"April 26, 2018",1.1,4.0.3 and up
7,Infinite Painter,ART_AND_DESIGN,4.1,36815,29M,"1,000,000+",Free,0,Everyone,Art & Design,"June 14, 2018",6.1.61.1,4.2 and up
8,Garden Coloring Book,ART_AND_DESIGN,4.4,13791,33M,"1,000,000+",Free,0,Everyone,Art & Design,"September 20, 2017",2.9.2,3.0 and up
9,Kids Paint Free - Drawing Fun,ART_AND_DESIGN,4.7,121,3.1M,"10,000+",Free,0,Everyone,Art & Design;Creativity,"July 3, 2018",2.8,4.0.3 and up
